In [1]:
!pip install tokenizers

In [ ]:
# Load dataset locally 
import pandas as pd
from pathlib import Path

# Set your data directory path
data_dir = Path("data/processed")

# Only need training set for tokenization 
train_df= pd.read_csv(data_dir / "train.csv")

In [2]:
# Load from Google Drive (if running on Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Load from Google Drive (if running on Colab)
import pandas as pd
from pathlib import Path

# Set your data directory path
data_dir = Path(
    "/content/drive/MyDrive/SMS-Spam-Detection/data/processed"
)

train = pd.read_csv(data_dir / "train.csv")

In [4]:
# Check the first few rows of the training dataset
train.head()

,TEXT,LABEL
0,Just got up. have to be out of the room very s...,0
1,Yes. I come to nyc for audiitions and am tryin...,0
2,"Sorry, I'll call later in night",0
3,Ok try to do week end course in coimbatore.,0
4,U still going to the mall?,0


In [5]:
"""We just need the text data for tokenization, so we can extract the "TEXT" column from the training dataset and convert it to a list. 
This will give us a list of all the text messages in the training set, 
which we can then use for tokenization.
"""

training_set = train["TEXT"].tolist()
print(f"5 sample text messages from the training set:\n{training_set[:5]}")

5 sample text messages from the training set:
["Just got up. have to be out of the room very soon. …. i hadn't put the clocks back til at 8 i shouted at everyone to get up and then realised it was 7. wahay. another hour in bed.", 'Yes. I come to nyc for audiitions and am trying to relocate.', "Sorry, I'll call later in night", 'Ok try to do week end course in coimbatore.', 'U still going to the mall?']


In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
# Need to construct tokenizer with BPE model class and specify the unknown token
# Tokenizer class is a wrapper around the BPE model that provides additional functionality 
# for tokenization and decoding.
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Prefix space is set to False because we don't want 
# to add a space before the first token in the input text.
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

# Need to construct a trainer for the BPE model.
trainer = BpeTrainer(vocab_size=2000, min_frequency=2, special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], initial_alphabet=ByteLevel.alphabet(), show_progress=True)


In [7]:
# Train tokenizer on the training set
tokenizer.train_from_iterator(training_set, trainer=trainer)

In [8]:
print(tokenizer.get_vocab_size())
print(tokenizer.get_vocab())

2000
{'Ġpick': 900, '÷': 183, 'Ġinto': 1412, '"': 5, 'ox': 876, 'king': 534, 'Ġmuch': 707, 'ord': 907, 'Ý': 157, 'Ġprize': 729, 'î': 174, 'work': 1391, 'ich': 939, 'ĠFOR': 1746, 'Ġprob': 813, 'Ġcode': 1794, '....': 774, 'ress': 965, '#&': 472, 'laim': 556, 'Ġnum': 690, 'Ġdraw': 1111, 'ĠThen': 1731, 'iday': 940, 'come': 1521, 'op': 364, 'os': 479, 'Ġshould': 847, 'more': 1779, 'Ġyears': 1761, '»': 123, 'iy': 1522, 'ater': 582, 'ART': 1558, 'Ġav': 1241, 'ix': 1398, 'side': 1584, 'i': 76, 'ĠD': 374, 'ĠXX': 1785, 'lect': 909, 'orry': 602, 'ter': 470, '99': 770, 'cc': 644, 'Ġto': 275, 'ones': 1134, 'Ġthen': 531, 'ĠMobile': 941, '24': 1471, 'Ġ3': 507, 'Ġsm': 593, 'Ġwe': 347, 'Ġbecause': 1306, 'Ġdays': 1255, 'Ġso': 405, 'Ġcos': 1090, 'ĠOff': 1989, 'orn': 1955, "'ve": 764, 'ĠS': 322, 'Ġcan': 377, 'Ġ200': 1605, 'ĠBox': 1896, 'Ġjob': 1098, 'uh': 1644, 'kia': 1003, 'Ġi': 272, 'Ġall': 492, 'Ġres': 1095, 'yt': 1948, '=': 32, 'ift': 1456, 'll': 274, 'Ġha': 308, 'Ġlunch': 1249, 'ets': 1799, '26': 157

In [9]:
for token in ["[UNK]", "[PAD]", "[SOS]", "[EOS]"]:
    print(token, tokenizer.token_to_id(token))

[UNK] 0
[PAD] 1
[SOS] 2
[EOS] 3


In [10]:
# Lets test our tokenizer

tokenized = tokenizer.encode("Hello, world! This is not a spam!")

print(f"Tokenized output: {tokenized.tokens}")
print(f"Token IDs: {tokenized.ids}")
print(f"Attention Mask: {tokenized.attention_mask}")

Tokenized output: ['Hello', ',', 'Ġworld', '!', 'ĠThis', 'Ġis', 'Ġnot', 'Ġa', 'Ġsp', 'am', '!']
Token IDs: [1336, 15, 1553, 4, 1010, 324, 396, 261, 524, 360, 4]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
# Post-processing is needed to add special tokens to the input sequence.
from tokenizers.processors import TemplateProcessing

tokenizer.post_processor = TemplateProcessing(
    single="[SOS] $A [EOS]",
    special_tokens=[
        ("[SOS]", tokenizer.token_to_id("[SOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]"))
    ]
)

In [14]:
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
tokenizer.decoder = ByteLevelDecoder()
tokenized = tokenizer.encode("Hello, world! This is not a spam!")
print(f"Tokenized output: {tokenized.tokens}")
print(f"Token IDs: {tokenized.ids}")
print(f"Decoded output: {tokenizer.decode(tokenized.ids)}")

Tokenized output: ['[SOS]', 'Hello', ',', 'Ġworld', '!', 'ĠThis', 'Ġis', 'Ġnot', 'Ġa', 'Ġsp', 'am', '!', '[EOS]']
Token IDs: [2, 1336, 15, 1553, 4, 1010, 324, 396, 261, 524, 360, 4, 3]
Decoded output: Hello, world! This is not a spam!


In [ ]:
# Save tokenizer locally
from pathlib import Path

tokenizer_dir = Path("tokenizer")
tokenizer_dir.mkdir(exist_ok=True)
tokenizer.save(str(tokenizer_dir / "tokenizer.json"))

In [15]:
# Save tokenizer to colab drive
tokenizer_dir = Path("/content/drive/MyDrive/SMS-Spam-Detection/tokenizer")
tokenizer_dir.mkdir(exist_ok=True)
tokenizer.save(str(tokenizer_dir / "tokenizer.json"))